# Phase B Teacher — RoBERTa-large on FiNER-ORD (Colab T4)

Runs `configs/phase_b_teacher.yaml` (3 seeds × up to 30 epochs, fp16, early stopping).

T4 wall-clock estimate: ~45–90 min per seed with early stopping. Budget a Colab Pro session; free tier may time out mid-run.

**Before you start:**
- Runtime → Change runtime type → T4 GPU.
- Repo must be reachable from Colab (Drive mount or GitHub clone in the next cell).
- `WANDB_API_KEY` must be set (Colab secret preferred, inline fallback).

In [ ]:
!nvidia-smi

In [5]:
from google.colab import runtime
runtime.unassign()

In [ ]:
# # Pick ONE approach below.

# Option A: Google Drive (recommended while the repo is local-only)
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/drive/MyDrive/dunedain_ner'  # adjust to your Drive layout
assert os.path.isdir(REPO_DIR), f'Repo not found at {REPO_DIR}. Upload or sync it first.'
%cd $REPO_DIR

# Option B: GitHub clone (uncomment if pushed)
# !git clone https://github.com/<you>/dunedain_ner.git /content/dunedain_ner
# %cd /content/dunedain_ner

!pwd && ls

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    import getpass
    os.environ['WANDB_API_KEY'] = getpass.getpass('WANDB_API_KEY: ')

!wandb login --relogin $WANDB_API_KEY

In [ ]:
# Smoke test first (~3-5 min) to catch plumbing issues before the long run.
!python -m src.train --config configs/smoke_test.yaml

In [ ]:
# Phase B teacher - 3 seeds x up to 30 epochs. Long-running.
!python -m src.train --config configs/phase_b_teacher.yaml

In [ ]:
import pandas as pd
df = pd.read_csv('results/results.csv')
phase_b = df[df['notes'].str.contains('phase_b_teacher', na=False)]
print(phase_b.to_string(index=False))
print()
print('Mean entity F1:', phase_b['test_entity_f1'].astype(float).mean())
print('Std  entity F1:', phase_b['test_entity_f1'].astype(float).std())

In [ ]:
import json
with open('results/phase_b_teacher_aggregate.json') as f:
    agg = json.load(f)
print(json.dumps(agg['test_metrics'], indent=2))